# AMML Ranking Competition 2026 — solution

**Preferred:** Kaggle Notebook **GPU**.  
**Metric:** NDCG@10  

Pipeline (course-aligned):
1. Lexical/structural features (BM25, TF-IDF, overlap, …)
2. **GroupKFold OOF** fine-tune of `rubert-tiny2` cross-encoder → `ce_score` feature
3. CatBoost **YetiRank** on features + CE (no post-hoc leaky blend)

Fine-tuning uses **only** competition train labels.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()
print('platform:', platform.platform())
print('python:', sys.version)
print('on_kaggle:', ON_KAGGLE)

if ON_KAGGLE:
    OUT = Path('/kaggle/working')
    MODELS = OUT / 'models'
    candidates = [
        Path('/kaggle/input/amml-ranking-competition-2026'),
        Path('/kaggle/input/competitions/amml-ranking-competition-2026'),
    ]
    DATA = next((p for p in candidates if (p / 'train.csv').exists()), None)
    if DATA is None:
        hits = list(Path('/kaggle/input').rglob('train.csv'))
        DATA = hits[0].parent if hits else candidates[0]
    CODE_DS = Path('/kaggle/input/amml-ranking-2026-code')
    if (CODE_DS / 'src').exists():
        ROOT = CODE_DS
    else:
        ROOT = OUT / 'amml-ranking-competition-2026'
        if not ROOT.exists():
            subprocess.check_call([
                'git', 'clone', '--depth', '1',
                'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git',
                str(ROOT),
            ])
    try:
        import rank_bm25  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25==0.2.2'])
    import catboost, torch
    print('catboost', catboost.__version__, 'torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
else:
    ROOT = Path('.').resolve()
    DATA = ROOT / 'data'
    OUT = ROOT / 'results'
    MODELS = ROOT / 'models'

sys.path.insert(0, str(ROOT))
# Prefer freshly cloned/repo code over stale dataset copy when both exist
if ON_KAGGLE:
    repo = Path('/kaggle/working/amml-ranking-competition-2026')
    if not repo.exists():
        subprocess.call([
            'git', 'clone', '--depth', '1',
            'https://github.com/VenistorAlishi/amml-ranking-competition-2026.git',
            str(repo),
        ])
    if (repo / 'src' / 'ce_train.py').exists():
        sys.path.insert(0, str(repo))
        ROOT = repo
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
print('ROOT', ROOT)
print('DATA', DATA)
assert (DATA / 'train.csv').exists(), list(Path('/kaggle/input').glob('*')) if ON_KAGGLE else DATA

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.preprocessing import prepare_frame
from src.features import assemble_features
from src.train import cross_validate, train_full, predict, get_feature_cols
from src.ce_train import oof_cross_encoder_scores
from src.metrics import mean_ndcg_at_k

train = prepare_frame(pd.read_csv(DATA / 'train.csv'))
test = prepare_frame(pd.read_csv(DATA / 'test.csv'))
print(train.shape, test.shape)

print('Building lexical/structural features...')
feat_tr, feat_te = assemble_features(train, test)

ce_epochs = 2 if torch.cuda.is_available() else 1
ce_bs = 64 if torch.cuda.is_available() else 16
print(f'CE OOF fine-tune epochs={ce_epochs} bs={ce_bs}')
ce_oof, ce_te = oof_cross_encoder_scores(
    train, test,
    models_dir=MODELS,
    epochs=ce_epochs,
    batch_size=ce_bs,
    max_len=192,
    n_splits=5,
)

# Add CE as features (+ within-query rank/z)
feat_tr = feat_tr.copy()
feat_te = feat_te.copy()
feat_tr['ce_score'] = ce_oof
feat_te['ce_score'] = ce_te
for df in (feat_tr, feat_te):
    g = df.groupby('query_id')['ce_score']
    df['ce_score_qrank'] = g.rank(method='average', ascending=False)
    mean = g.transform('mean')
    std = g.transform('std').replace(0, np.nan)
    df['ce_score_qz'] = ((df['ce_score'] - mean) / std).fillna(0.0)

labels = train['relevance']
print('CatBoost YetiRank with CE features...')
cv, oof = cross_validate(feat_tr, labels, n_splits=5)
model = train_full(feat_tr, labels, MODELS, best_iterations=1500)
feature_cols = get_feature_cols(feat_tr.assign(relevance=labels))
pred = predict(model, feat_te, feature_cols)

scored = train[['id', 'query_id', 'relevance']].copy()
scored['prediction'] = oof
print('OOF NDCG@10', mean_ndcg_at_k(scored), 'CV', cv)

sub = test[['id']].copy()
sub['prediction'] = pred
sub.to_csv(OUT / 'submission.csv', index=False)
print('wrote', OUT / 'submission.csv', len(sub))
sub.head()